# DiffBind Differential Accessibility Analysis

## 1. Load Libraries


In [1]:
library(DiffBind)
library(ChIPseeker)
library(GenomicFeatures)
library(TxDb.Mmusculus.UCSC.mm10.knownGene)
library(org.Mm.eg.db)
library(dplyr)

Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘pac

## 2. Configuration

In [ ]:
BAM_DIR  <- "/coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/results/bwa/merged_library"
PEAK_DIR <- file.path(BAM_DIR, "macs2", "broad_peak")
OUT_DIR  <- "/coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/diffbind_results"

CONDITIONS   <- c("GFP", "TFF1")  
N_REPS       <- 3                  
CONTROL_COND <- "GFP"               # reference level for contrast
TREAT_COND   <- "TFF1"              # treatment level

BAM_SUFFIX  <- ".mLb.clN.sorted.bam"      
PEAK_SUFFIX <- ".mLb.clN_peaks.broadPeak" 

SCORE_COL   <- 8      # broadPeak col 8 = signalValue (MACS2 fold enrichment)
MIN_OVERLAP <- 2      # consensus: peak must appear in >= N samples
SUMMITS     <- FALSE  # FALSE for broadPeak

ANALYSIS_METHOD <- DBA_DESEQ2  

FDR_THRESH <- 0.05   # FDR cutoff for significance

TSS_UP   <- 3000   # bp upstream of TSS for promoter annotation
TSS_DOWN <- 3000   # bp downstream of TSS for promoter annotation

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
cat("Output directory:", OUT_DIR, "\n")

Output directory: /coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/diffbind_results 


## 3. Build DiffBind Sample Sheet



In [3]:
sample_sheet <- do.call(rbind, lapply(CONDITIONS, function(cond) {
    do.call(rbind, lapply(seq_len(N_REPS), function(rep) {
        sample_id <- sprintf("%s_REP%d", cond, rep)
        bam_file  <- file.path(BAM_DIR,  paste0(sample_id, BAM_SUFFIX))
        peak_file <- file.path(PEAK_DIR, paste0(sample_id, PEAK_SUFFIX))
        data.frame(
            SampleID   = sample_id,
            Condition  = cond,
            Replicate  = rep,
            bamReads   = bam_file,
            Peaks      = peak_file,
            PeakCaller = "bed",   # DiffBind reads broadPeak as BED
            stringsAsFactors = FALSE
        )
    }))
}))

# Verify all files exist before proceeding
missing <- c(
    sample_sheet$bamReads[!file.exists(sample_sheet$bamReads)],
    sample_sheet$Peaks[!file.exists(sample_sheet$Peaks)]
)
if (length(missing) > 0) {
    stop("Missing files — check BAM_DIR / PEAK_DIR paths:\n",
         paste(missing, collapse = "\n"))
}

cat("Sample sheet:\n")
print(sample_sheet[, c("SampleID", "Condition", "Replicate")])
cat("\nAll", nrow(sample_sheet), "BAM and peak files found.\n")

Sample sheet:
   SampleID Condition Replicate
1  GFP_REP1       GFP         1
2  GFP_REP2       GFP         2
3  GFP_REP3       GFP         3
4 TFF1_REP1      TFF1         1
5 TFF1_REP2      TFF1         2
6 TFF1_REP3      TFF1         3

All 6 BAM and peak files found.


## 4. Create DiffBind Object

- `bRemoveM = TRUE` — exclude mitochondrial reads
- Duplicate reads already removed by nf-core pipeline (Picard)

In [4]:
dba_obj <- dba(
    sampleSheet = sample_sheet,
    scoreCol    = SCORE_COL,
    bRemoveM    = TRUE
)

print(dba_obj)

GFP_REP1   GFP  1 bed

GFP_REP2   GFP  2 bed

GFP_REP3   GFP  3 bed

TFF1_REP1   TFF1  1 bed

TFF1_REP2   TFF1  2 bed

TFF1_REP3   TFF1  3 bed



6 Samples, 6520 sites in matrix (19294 total):
         ID Condition Replicate Intervals
1  GFP_REP1       GFP         1      8127
2  GFP_REP2       GFP         2      9168
3  GFP_REP3       GFP         3      6870
4 TFF1_REP1      TFF1         1      4872
5 TFF1_REP2      TFF1         2      6932
6 TFF1_REP3      TFF1         3      1838


## 5. Count Reads in Consensus Peaks

In [5]:
dba_obj <- dba.count(
    dba_obj,
    bUseSummarizeOverlaps = TRUE,
    minOverlap            = MIN_OVERLAP,
    summits               = SUMMITS,
    score                 = DBA_SCORE_READS,
    bRemoveDup            = FALSE
)

print(dba_obj)

licate    Reads FRiP
1  GFP_REP1       GFP         1 60067594 0.01
2  GFP_REP2       GFP         2 50547114 0.01
3  GFP_REP3       GFP         3 55881993 0.01
4 TFF1_REP1      TFF1         1 70062452 0.01
5 TFF1_REP2      TFF1         2 65292370 0.01
6 TFF1_REP3      TFF1         3 66931089 0.00


In [13]:
pdf(file = "diffbind_results/0_figures/pre_normalization_sample_pca.pdf", width = 8, height = 8)
pre_normlaization_sample_pca <- dba.plotPCA(dba_obj, attributes=DBA_FACTOR, label=DBA_ID)
dev.off()

agg_record_6bae652f585d8 
                       2

In [14]:
# Peak overlap heatmap (before counting)
pdf(file = "diffbind_results/0_figures/peak_overlap_heatmap_beforecounting.pdf", width = 8, height = 8)
dba.plotHeatmap(dba_obj)
dev.off()

agg_record_6bae6791961e9 
                       2

In [17]:
cts <- dba.peakset(dba_obj, bRetrieve=TRUE, DataType=DBA_DATA_FRAME)
cat('Peaks:', nrow(cts), '\n')
cat('Range of counts across peaks:\n')
print(summary(rowSums(cts[, -(1:3)])))
cat('Samples with zero total counts:\n')
print(colSums(cts[, -(1:3)]))

Peaks: 6514 
Range of counts across peaks:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   42.0   169.0   247.0   347.2   381.0 25385.0 
Samples with zero total counts:
 GFP_REP1  GFP_REP2  GFP_REP3 TFF1_REP1 TFF1_REP2 TFF1_REP3 
   408116    414719    384274    363812    401778    288741 


## 6. Normalize

`DBA_LIBSIZE_PEAKREADS` normalizes to reads falling within peaks only (not total library size) — recommended for ATAC-seq.

In [ ]:
dba_obj <- dba.normalize(
    dba_obj,
    bSubControl = FALSE  
)

# Inspect normalization factors
norm <- dba.normalize(dba_obj, bRetrieve = TRUE)
norm

$norm.method
[1] "lib"

$norm.factors
[1] 0.9772846 0.8223888 0.9091859 1.1398984 1.0622904 1.0889519

$lib.method
[1] "full"

$lib.sizes
[1] 60067594 50547114 55881993 70062452 65292370 66931089

$control.subtract
[1] TRUE

$filter.value
[1] 1

In [19]:
info <- dba.show(dba_obj)
normlibs <- cbind(FullLibSize=norm$lib.sizes, NormFacs=norm$norm.factors,
                  NormLibSize=round(norm$lib.sizes/norm$norm.factors))

rownames(normlibs) <- info$ID
normlibs

,FullLibSize,NormFacs,NormLibSize
GFP_REP1,60067594,0.9772846,61463769
GFP_REP2,50547114,0.8223888,61463769
GFP_REP3,55881993,0.9091859,61463769
TFF1_REP1,70062452,1.1398984,61463769
TFF1_REP2,65292370,1.0622904,61463769
TFF1_REP3,66931089,1.0889519,61463769


## 7. Contrast & Analysis

**Fold convention: positive = TFF1-enriched, negative = GFP-enriched**

In [ ]:
dba_obj <- dba.contrast(
    dba_obj,
    design   = ~ Condition,
    contrast = c("Condition", TREAT_COND, CONTROL_COND)
)

dba.show(dba_obj, bContrasts = TRUE)

Computing results names...



,Factor,Group,Samples,Group2,Samples2
,<chr>,<chr>,<chr>,<chr>,<chr>
1,Condition,TFF1,3,GFP,3


In [ ]:
# Run differential analysis with DESeq2
dba_obj <- dba.analyze(
    dba_obj,
    method     = ANALYSIS_METHOD,
    bBlacklist = FALSE,
    bGreylist  = FALSE
)

dba.show(dba_obj, bContrasts = TRUE)

Analyzing...

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates



,Factor,Group,Samples,Group2,Samples2,DB.DESeq2
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,Condition,TFF1,3,GFP,3,1424


### Correlation Heatmap

In [22]:
pdf(file = "diffbind_results/0_figures/samplecorrelationheatmap.pdf", width = 8, height = 8)
dba.plotHeatmap(
    dba_obj,
    ColAttributes = DBA_CONDITION,
    main = "Sample Correlation Heatmap"
)
dev.off()

agg_record_6bae642e6c787 
                       2

## 9. Extract Results

In [ ]:
res_all <- dba.report(dba_obj, method=DBA_DESEQ2, th=1)

sig_d <- res_all[!is.na(res_all$FDR) & res_all$FDR < FDR_THRESH]

cat(sprintf('\nDESeq2: %d total | %d sig | %d %s-enriched | %d %s-enriched\n',
    length(res_all), length(sig_d),
    sum(sig_d$Fold > 0), TREAT_COND,
    sum(sig_d$Fold < 0), CONTROL_COND))

save_bed <- function(gr, path) {
    if (length(gr) > 0)
        write.table(data.frame(chr=seqnames(gr), start=start(gr)-1L, end=end(gr)),
                    path, sep='\t', row.names=FALSE, col.names=FALSE, quote=FALSE)
}
save_bed(sig_d[sig_d$Fold > 0], file.path(OUT_DIR, paste0('diffbind_', TREAT_COND,    '_enriched.bed')))
save_bed(sig_d[sig_d$Fold < 0], file.path(OUT_DIR, paste0('diffbind_', CONTROL_COND, '_enriched.bed')))


DESeq2: 6514 total | 1424 sig | 1 TFF1-enriched | 1423 GFP-enriched


In [24]:
#par(mfrow=c(3,4))
pdf(file = "diffbind_results/0_figures/tff1vsgfp_MAplot.pdf", width = 8, height = 8)
dba.plotMA(dba_obj, method=DBA_DESEQ2)
dev.off()

agg_record_6bae6767e3414 
                       2

In [25]:
pdf(file = "diffbind_results/0_figures/tff1vsgfp_volcanoplot.pdf", width = 8, height = 8)
dba.plotVolcano(dba_obj, method=DBA_DESEQ2)
dev.off()

agg_record_6bae64ef2465a 
                       2

## 9. ChIPseeker Annotation (mm10)

In [ ]:
txdb <- TxDb.Mmusculus.UCSC.mm10.knownGene

annotate_gr <- function(gr, txdb, tss_up, tss_down) {
    anno <- annotatePeak(
        gr,
        TxDb      = txdb,
        tssRegion = c(-tss_up, tss_down),
        annoDb    = 'org.Mm.eg.db'
    )
    df <- as.data.frame(anno)
    names(df)[names(df) == 'p-value'] <- 'pvalue'
    df
}

anno_df <- annotate_gr(res_all, txdb, TSS_UP, TSS_DOWN)

anno_df$significant <- !is.na(anno_df$FDR) & anno_df$FDR < FDR_THRESH
anno_df$direction   <- ifelse(!anno_df$significant, 'NS',
                       ifelse(anno_df$Fold > 0, TREAT_COND, CONTROL_COND))

cat(sprintf('Total peaks annotated: %d (%d significant)\n', nrow(anno_df), sum(anno_df$significant)))
print(head(anno_df[, c('seqnames', 'start', 'end', 'Fold', 'FDR', 'annotation', 'SYMBOL', 'distanceToTSS')]))

if (sum(anno_df$significant) > 0) {
    anno_all_obj <- annotatePeak(res_all, TxDb=txdb, tssRegion=c(-TSS_UP, TSS_DOWN))
    pdf(file = "diffbind_results/0_figures/all_peaks_pie_plot.pdf", width = 8, height = 8)
    par(oma = c(0, 0, 2, 0))
    plotAnnoPie(anno_all_obj)
    mtext("All Peaks", outer = TRUE, side = 3, line = 0.5, cex = 1.4, font = 2)
    dev.off()

    anno_sig_obj <- annotatePeak(sig_d, TxDb=txdb, tssRegion=c(-TSS_UP, TSS_DOWN))
    pdf(file = "diffbind_results/0_figures/sig_peaks_pie_plot.pdf", width = 8, height = 8)
    par(oma = c(0, 0, 2, 0))
    plotAnnoPie(anno_sig_obj)
    mtext("Significant Peaks", outer = TRUE, side = 3, line = 0.5, cex = 1.4, font = 2)
    dev.off()
} else {
    anno_all_obj <- annotatePeak(res_all, TxDb=txdb, tssRegion=c(-TSS_UP, TSS_DOWN))
    pdf(file = "diffbind_results/0_figures/all_peaks_pie_plot.pdf", width = 8, height = 8)
    par(oma = c(0, 0, 2, 0))
    plotAnnoPie(anno_all_obj)
    mtext("All Peaks", outer = TRUE, side = 3, line = 0.5, cex = 1.4, font = 2)
    dev.off()
}

>> preparing features information...		 2026-03-24 10:27:44 
>> Using Genome: mm10 ...
>> identifying nearest features...		 2026-03-24 10:27:44 
>> calculating distance from peak to TSS...	 2026-03-24 10:27:45 
>> assigning genomic annotation...		 2026-03-24 10:27:45 
>> Using Genome: mm10 ...
>> Using Genome: mm10 ...
>> adding gene annotation...			 2026-03-24 10:27:47 


'select()' returned 1:many mapping between keys and columns



>> assigning chromosome lengths			 2026-03-24 10:27:48 
>> done...					 2026-03-24 10:27:48 
Total peaks annotated: 6454 (1423 significant)
        seqnames     start       end       Fold          FDR        annotation
1           chr9 124422628 124425822 -0.7689315 1.415592e-05            5' UTR
2          chr14  82065236  82065650 -1.1743464 2.623785e-05 Distal Intergenic
3 chrUn_JH584304     47363     50666 -0.2996125 7.213982e-05 Distal Intergenic
4           chr3 152478937 152479202 -1.2944618 1.831334e-04  Promoter (<=1kb)
5          chr16  62846948  62847283 -1.0806319 3.156997e-04  Promoter (<=1kb)
6           chr9 108305938 108306337 -1.0433561 5.669020e-04  Promoter (<=1kb)
    SYMBOL distanceToTSS
1  Ppp2r3d         51076
2  Gm32455       2009640
3 Pisd-ps3          9001
4      Ak5          -422
5   Arl13b             0
6     Tcta             0
>> preparing features information...		 2026-03-24 10:27:48 
>> Using Genome: mm10 ...
>> identifying nearest features...		 2026-03-2

agg_record_6bae637094650 
                       2

## 10. Save Outputs

In [ ]:
treat_sig   <- anno_df[!is.na(anno_df$direction) & anno_df$direction == TREAT_COND,   ]
control_sig <- anno_df[!is.na(anno_df$direction) & anno_df$direction == CONTROL_COND, ]
all_sig     <- anno_df[!is.na(anno_df$direction) & anno_df$direction != 'NS',          ]

write.csv(anno_df,     file.path(OUT_DIR, 'diffbind_all_results.csv'),                          row.names=FALSE)
write.csv(all_sig,     file.path(OUT_DIR, 'diffbind_sig_results.csv'),                          row.names=FALSE)
write.csv(treat_sig,   file.path(OUT_DIR, paste0('diffbind_', TREAT_COND,   '_enriched.csv')), row.names=FALSE)
write.csv(control_sig, file.path(OUT_DIR, paste0('diffbind_', CONTROL_COND, '_enriched.csv')), row.names=FALSE)

saveRDS(dba_obj, file = file.path(OUT_DIR, 'dba_object.rds'))

cat(sprintf('Saved CSVs in: %s\n  diffbind_all_results.csv          (%d peaks)\n  diffbind_sig_results.csv          (%d significant)\n  diffbind_%s_enriched.csv   (%d peaks)\n  diffbind_%s_enriched.csv    (%d peaks)\n',
    OUT_DIR, nrow(anno_df), nrow(all_sig),
    TREAT_COND,   nrow(treat_sig),
    CONTROL_COND, nrow(control_sig)))
cat(sprintf('BED files: diffbind_%s_enriched.bed, diffbind_%s_enriched.bed\n', TREAT_COND, CONTROL_COND))
cat(sprintf('DiffBind object saved: dba_object.rds\n'))

Saved CSVs in: /coh_labs/mvandenbrink/users/pkaur/6_tff1/2_bulk_atac/atacseq_pipeline/0_broadpeak/diffbind_results
  diffbind_all_results.csv          (6454 peaks)
  diffbind_sig_results.csv          (1423 significant)
  diffbind_TFF1_enriched.csv   (1 peaks)
  diffbind_GFP_enriched.csv    (1422 peaks)
BED files: diffbind_TFF1_enriched.bed, diffbind_GFP_enriched.bed
DiffBind object saved: dba_object.rds


## 15. Session Info

In [30]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 9.7 (Blue Onyx)

Matrix products: default
BLAS/LAPACK: /coh_labs/mvandenbrink/users/pkaur/miniconda3/envs/bulkatac/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: America/Phoenix
tzcode source: system (glibc)

attached base packages:
[1] grid      stats4    stats     graphics  grDevices utils     datasets 
[8] methods   base     

other attached packages:
 [1] dplyr_1.1.4                              
 [2] org.Mm.eg.db_3.17.0                      
 [3] TxDb.Mmusculus.UCSC.mm10.knownGene_3.10.0
 [4] GenomicFeatures_1.52.1                   
 [5] AnnotationDbi_1.62.2          